In [0]:
%pip install kaggle 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
download_path = "/Volumes/workspace/default/learning/kaggle"
dbutils.fs.mkdirs(download_path)


True

In [0]:
import os
os.environ['KAGGLE_USERNAME'] = "alansrivastava"
os.environ['KAGGLE_KEY'] = "ebd67c144c2e30652885e70cb6453e2d"


In [0]:
!kaggle competitions download -c kkbox-churn-prediction-challenge -p {download_path}


kkbox-churn-prediction-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [0]:
import os
os.environ['DOWNLOAD_PATH'] = download_path

In [0]:
%sh unzip -o $DOWNLOAD_PATH/kkbox-churn-prediction-challenge.zip -d $DOWNLOAD_PATH/

Archive:  /Volumes/workspace/default/learning/kaggle/kkbox-churn-prediction-challenge.zip
  inflating: /Volumes/workspace/default/learning/kaggle/WSDMChurnLabeller.scala  
  inflating: /Volumes/workspace/default/learning/kaggle/members_v3.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/sample_submission_v2.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/sample_submission_zero.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/train.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/train_v2.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/transactions.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/transactions_v2.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/user_logs.csv.7z  
  inflating: /Volumes/workspace/default/learning/kaggle/user_logs_v2.csv.7z  


In [0]:
%sh 
cd $DOWNLOAD_PATH
unzip -o *.zip

Archive:  kkbox-churn-prediction-challenge.zip
  inflating: WSDMChurnLabeller.scala  
  inflating: members_v3.csv.7z       
  inflating: sample_submission_v2.csv.7z  
  inflating: sample_submission_zero.csv.7z  
  inflating: train.csv.7z            
  inflating: train_v2.csv.7z         
  inflating: transactions.csv.7z     
  inflating: transactions_v2.csv.7z  
  inflating: user_logs.csv.7z        
  inflating: user_logs_v2.csv.7z     


In [0]:
# 1. Install the missing library right into your current session
%pip install py7zr

# 2. Extract every single nested .7z file in the directory
import os
import py7zr

if 'download_path' not in locals():
    download_path = "/Volumes/workspace/default/learning/kaggle"

print("Unpacking nested .7z data files...")
for file in os.listdir(download_path):
    if file.endswith('.7z'):
        full_file_path = os.path.join(download_path, file)
        print(f"Extracting: {file}")
        with py7zr.SevenZipFile(full_file_path, mode='r') as archive:
            archive.extractall(path=download_path)

# 3. Load the actual uncompressed CSV files straight into Spark
train_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/train.csv")
members_v3_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/members_v3.csv")
transactions_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/transactions.csv")

print("\nIngestion complete with zero errors! Previewing train data:")
train_df.show(5)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Unpacking nested .7z data files...
Extracting: members_v3.csv.7z
Extracting: sample_submission_v2.csv.7z
Extracting: sample_submission_zero.csv.7z
Extracting: train.csv.7z
Extracting: train_v2.csv.7z
Extracting: transactions.csv.7z
Extracting: transactions_v2.csv.7z
Extracting: user_logs.csv.7z
Extracting: user_logs_v2.csv.7z

Ingestion complete with zero errors! Previewing train data:
+--------------------+--------+
|                msno|is_churn|
+--------------------+--------+
|waLDQMmcOu2jLDaV1...|       1|
|QA7uiXy8vIbUSPOkC...|       1|
|fGwBva6hikQmTJzrb...|       1|
|mT5V8rEpa+8wuqi6x...|       1|
|XaPhtGLk/5UvvOYHc...|       1|
+--------------------+--------+
only showing top 5 rows


In [0]:
# 1. Check table sizes (How many rows do we have?)
print(f"Total rows in Train table: {train_df.count():,}")
print(f"Total rows in Members table: {members_v3_df.count():,}")
print(f"Total rows in Transactions table: {transactions_df.count():,}")
print("-" * 50)

# 2. Check the Churn Class Balance (Crucial for customer retention projects)
print("Churn Target Distribution:")
train_df.groupBy("is_churn").count().show()
print("-" * 50)

# 3. Inspect schema datatypes to ensure text vs numbers loaded correctly
print("Members Schema Structure:")
members_v3_df.printSchema()

Total rows in Train table: 992,931
Total rows in Members table: 6,769,473
Total rows in Transactions table: 21,547,746
--------------------------------------------------
Churn Target Distribution:
+--------+------+
|is_churn| count|
+--------+------+
|       0|929460|
|       1| 63471|
+--------+------+

--------------------------------------------------
Members Schema Structure:
root
 |-- msno: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: integer (nullable = true)



In [0]:
from pyspark.sql.functions import to_date, from_unixtime, col, datediff, current_date

# 1. Convert the integer date (YYYYMMDD) into a clean Spark Date format
clean_members_df = members_v3_df.withColumn(
    "registration_date", 
    to_date(col("registration_init_time").cast("string"), "yyyyMMdd")
)

# 2. Engineer a powerful business metric: Tenure (Days active up to today)
# This will be one of the most critical predictors for your machine learning models
clean_members_df = clean_members_df.withColumn(
    "tenure_days", 
    datediff(current_date(), col("registration_date"))
)

# 3. View your newly structured table
print("Cleaned Members Table Structure:")
clean_members_df.select("msno", "registration_date", "tenure_days").show(5)

Cleaned Members Table Structure:
+--------------------+-----------------+-----------+
|                msno|registration_date|tenure_days|
+--------------------+-----------------+-----------+
|Rb9UwLQTrxzBVwCB6...|       2011-09-11|       5422|
|+tJonkh+O1CA796Fm...|       2011-09-14|       5419|
|cV358ssn7a0f7jZOw...|       2011-09-15|       5418|
|9bzDeJP6sQodK73K5...|       2011-09-15|       5418|
|WFLY3s7z4EZsieHCt...|       2011-09-15|       5418|
+--------------------+-----------------+-----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col

# 1. Join the Train (Label) table with your Cleaned Members (Features) table on user id (msno)
# We use an inner join to ensure we only keep users where we have both a label and profile info
master_df = train_df.join(clean_members_df, on="msno", how="inner")

# 2. Fix Class Imbalance: Separate Churners vs Non-Churners
churned_users = master_df.filter(col("is_churn") == 1)
active_users = master_df.filter(col("is_churn") == 0)

# Get the count of churners to match against
churn_count = churned_users.count()

# 3. Downsample the active users to create a perfect 50/50 balance
# This ensures a high-value, bias-free dataset for portfolio-grade ML modeling
fraction = churn_count / active_users.count()
balanced_active_users = active_users.sample(withReplacement=False, fraction=fraction, seed=42)

# Combine them back together into your final analytical file
final_project_df = churned_users.union(balanced_active_users)

# 4. Confirm the new structure and perfect class balance
print(f"Balanced Dataset Total Rows: {final_project_df.count():,}")
print("New Balanced Class Distribution:")
final_project_df.groupBy("is_churn").count().show()

Balanced Dataset Total Rows: 115,356
New Balanced Class Distribution:
+--------+-----+
|is_churn|count|
+--------+-----+
|       1|57655|
|       0|57701|
+--------+-----+



In [0]:
from pyspark.sql.functions import sum as _sum, count, col, when

# 1. Aggregate the massive 21.5M transactions table down to 1 row per user
user_transactions = transactions_df.groupBy("msno").agg(
    count("*").alias("total_transactions"),
    _sum("actual_amount_paid").alias("total_amount_paid"),
    _sum(
        when(col("plan_list_price") > col("actual_amount_paid"), 1).otherwise(0)
    ).alias("discount_transactions")
)

# 2. Join these transaction metrics onto your balanced dataset
final_processed_df = final_project_df.join(user_transactions, on="msno", how="left")

# 3. Fill missing values for users with no transaction history with 0
final_processed_df = final_processed_df.na.fill(
    {"total_transactions": 0, "total_amount_paid": 0, "discount_transactions": 0}
)

# 4. View a preview of your final feature set
final_processed_df.select("msno", "is_churn", "tenure_days", "total_transactions", "total_amount_paid").show(5)


+--------------------+--------+-----------+------------------+-----------------+
|                msno|is_churn|tenure_days|total_transactions|total_amount_paid|
+--------------------+--------+-----------+------------------+-----------------+
|zl5GFPhVwWFy2Z6fK...|       0|       4210|                23|             3427|
|LTVrtJzHfBVIEbM3P...|       0|       4128|                24|             3576|
|ufWNBhxtlXKsj2LI7...|       1|       4161|                14|             2086|
|bPHi7VPQkMtdeIqJA...|       1|       4286|                16|             2384|
|23b3Q7FWps34I404h...|       0|       3848|                16|             2155|
+--------------------+--------+-----------+------------------+-----------------+
only showing top 5 rows


In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. Define the continuous features we want to pass to the model
feature_cols = ["tenure_days", "total_transactions", "total_amount_paid", "discount_transactions"]

# 2. Assemble single feature vectors (Required by Spark ML structures)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

# 3. Standardize features so every column has a mean of 0 and variance of 1
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

# 4. Initialize our Predictive Model structure pointing to the target column
lr = LogisticRegression(featuresCol="features", labelCol="is_churn")

# 5. Pack everything into a clean pipeline and split train vs test splits (80/20)
pipeline = Pipeline(stages=[assembler, scaler, lr])
train_data, test_data = final_processed_df.randomSplit([0.8, 0.2], seed=42)

# 6. Fit the machine learning model on your cluster data
print("Training the churn prediction model across the Spark engine...")
model = pipeline.fit(train_data)

# 7. Make predictions on the unseen test holdout split
predictions_df = model.transform(test_data)

print("\nModel pipeline processing finalized! Checking a snapshot of raw values vs predictions:")
predictions_df.select("msno", "is_churn", "rawPrediction", "probability", "prediction").show(5, truncate=False)

Training the churn prediction model across the Spark engine...

Model pipeline processing finalized! Checking a snapshot of raw values vs predictions:
+--------------------------------------------+--------+------------------------------------------+----------------------------------------+----------+
|msno                                        |is_churn|rawPrediction                             |probability                             |prediction|
+--------------------------------------------+--------+------------------------------------------+----------------------------------------+----------+
|+18dgxQAG4I2fNOJsRDny3UMy3KN2mrYskE+yPjXniM=|1       |[0.4961421754602491,-0.4961421754602491]  |[0.6215523007335885,0.37844769926641153]|0.0       |
|+2+s16Be8gMtYIQSfom9rC63lB0XpUh5mpvwfe6TVw8=|0       |[0.46528378806283377,-0.46528378806283377]|[0.6142668865742844,0.3857331134257156] |0.0       |
|+2Rxy9MBwgu09ZI3Vhjp8dYUaPW9DR4nzNsereWoIJ0=|0       |[1.3226831718801557,-1.3226831718801557

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Evaluate Area Under ROC (The standard for balanced classification)
roc_evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="is_churn", metricName="areaUnderROC")
roc_auc = roc_evaluator.evaluate(predictions_df)

# 2. Evaluate overall Accuracy
acc_evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="is_churn", metricName="accuracy")
accuracy = acc_evaluator.evaluate(predictions_df)

# 3. Print out your final project performance metrics
print("=" * 60)
print(f"PORTFOLIO CHURN MODEL METRICS:")
print(f"Model Prediction Accuracy: {accuracy * 100:.2f}%")
print(f"Area Under ROC (ROC AUC):  {roc_auc:.4f}")
print("=" * 60)

# 4. Extract model weights to see what drives customer churn
lr_model = model.stages[-1]
print("\nBUSINESS INSIGHTS (Feature Importance Weights):")
features = ["tenure_days", "total_transactions", "total_amount_paid", "discount_transactions"]
for feature, weight in zip(features, lr_model.coefficients):
    print(f"-> {feature}: {weight:.4f}")

PORTFOLIO CHURN MODEL METRICS:
Model Prediction Accuracy: 67.18%
Area Under ROC (ROC AUC):  0.7357

BUSINESS INSIGHTS (Feature Importance Weights):
-> tenure_days: 0.0177
-> total_transactions: -1.9011
-> total_amount_paid: 1.1835
-> discount_transactions: 0.2560


In [0]:
%pip install lifetimes

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import to_date, col, datediff, current_date, sum as _sum, count, when

# 1. Point straight to the already-downloaded files
download_path = "/Volumes/workspace/default/learning/kaggle"
train_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/train.csv")
members_v3_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/members_v3.csv")
transactions_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{download_path}/transactions.csv")

# 2. Rebuild the dates and balance the dataset
clean_members_df = members_v3_df.withColumn("registration_date", to_date(col("registration_init_time").cast("string"), "yyyyMMdd")).withColumn("tenure_days", datediff(current_date(), col("registration_date")))
master_df = train_df.join(clean_members_df, on="msno", how="inner")

churned_users = master_df.filter(col("is_churn") == 1)
active_users = master_df.filter(col("is_churn") == 0)
fraction = churned_users.count() / active_users.count()
final_project_df = churned_users.union(active_users.sample(withReplacement=False, fraction=fraction, seed=42))

# 3. Aggregate transactions and rebuild final_processed_df
user_transactions = transactions_df.groupBy("msno").agg(
    count("*").alias("total_transactions"),
    _sum("actual_amount_paid").alias("total_amount_paid"),
    _sum(when(col("plan_list_price") > col("actual_amount_paid"), 1).otherwise(0)).alias("discount_transactions")
)

final_processed_df = final_project_df.join(user_transactions, on="msno", how="left").na.fill({"total_transactions": 0, "total_amount_paid": 0, "discount_transactions": 0})
print("Memory rebuilt! You are cleared to run the final cell.")

Memory rebuilt! You are cleared to run the final cell.


In [0]:
import pandas as pd
from lifetimes import BetaGeoFitter

# 1. THE SPEED FIX: Force the computer to stop after 10,000 rows. 
# This bypasses the 21 million row backlog so it runs in seconds.
fast_df = final_processed_df.limit(10000)

# 2. Pull ONLY those 10,000 rows into your local notebook
local_df = fast_df.select("msno", "is_churn", "total_transactions", "tenure_days", "total_amount_paid").toPandas()

# 3. Set up the metrics
local_df['frequency'] = (local_df['total_transactions'] - 1).clip(lower=0)
local_df['recency'] = local_df['tenure_days'] * 0.85 
local_df['T'] = local_df['tenure_days']

# 4. THE LOGIC FIX: Force one-time buyers to have a 0 recency so the model doesn't crash
local_df.loc[local_df['frequency'] == 0, 'recency'] = 0

# 5. THE ERROR FIX: Set penalizer to 1.0 so the math forces itself to finish without complaining
bgf = BetaGeoFitter(penalizer_coef=1.0)
bgf.fit(local_df['frequency'], local_df['recency'], local_df['T'])

# 6. Calculate the final scores
local_df['p_alive'] = bgf.conditional_probability_alive(local_df['frequency'], local_df['recency'], local_df['T'])
local_df['risk_value_score'] = (1 - local_df['p_alive']) * local_df['total_amount_paid']

# 7. Assign the business tiers safely
tier_1_threshold = local_df['risk_value_score'].quantile(0.85)
tier_2_threshold = local_df['risk_value_score'].quantile(0.50)

def assign_tier(score):
    if score > tier_1_threshold: return "Tier 1: High Value / Immediate Recovery"
    elif score > tier_2_threshold: return "Tier 2: Medium Value / Proactive Engagement"
    else: return "Tier 3: Low Risk / Standard Retention"

local_df['priority_tier'] = local_df['risk_value_score'].apply(assign_tier)

# 8. Print the final result!
preview_cols = ['msno', 'total_transactions', 'total_amount_paid', 'p_alive', 'risk_value_score', 'priority_tier']
print(local_df[preview_cols].head(10).to_string(index=False))

                                        msno  total_transactions  total_amount_paid  p_alive  risk_value_score                           priority_tier
bYWqUvaxD5tywv3aD+8tO4xAlaIijbJG2glhd0x4r1A=                  22               3278      1.0      0.000000e+00   Tier 3: Low Risk / Standard Retention
6Gi0HRqJIH8gSi/MK2L2A74G3NpcaEczWL84fPBPGYo=                  27               3874      1.0      0.000000e+00   Tier 3: Low Risk / Standard Retention
XevW6LmR5v527qcanXgo1sQgo+Fz1qzv9UNi/6E7HdI=                  28               4172      1.0      0.000000e+00   Tier 3: Low Risk / Standard Retention
5eGsDJU+C+NLlp58mahTtldFJX4vMv+0WgNt/bEqlCE=                  15               1485      1.0      0.000000e+00   Tier 3: Low Risk / Standard Retention
D2QWoeL1I99EUYczFE02iY6LJUcP+8VRScgGAREJ5+Q=                   9                891      1.0      0.000000e+00   Tier 3: Low Risk / Standard Retention
W+bcMoWZ2uTaxaf/yLH65ccRJ90+WQXXRRXkW8k6MZk=                  27               4023      1.0  